In [3]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle, os
import train

# ── paste these from your pipeline ───────────────────────────────────────────
from train import (
    MultimodalCNN, ConvAutoencoder, FusedClassifier,
    CLASSES, FEAT_COLS, TAB_DIM, LATENT_DIM, IMG_SIZE, DEVICE
)

OUT_DIR  = "/media/Ubunt_2/Project/pipeline_output"

In [6]:
train.__name__ = "__refit_scaler__"
exec(open("train.py").read())

[config] device=cuda  tabular_dim=9  latent_dim=128

══════════════════════════════════════════════════════════
  Stage 0 : Data & Feature Engineering
══════════════════════════════════════════════════════════
[data] 260201/300000 images found on disk
[features] dropped 0 rows with NaN features, 260201 remain
[data] train=208160  val=26020  test=26021
[plot] /media/Ubunt_2/Project/pipeline_output/feature_distributions.png

══════════════════════════════════════════════════════════
  Stage 0b : Hyperparameter Tuning
══════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════
  Stage 1 : Multimodal CNN  (image + tabular)
══════════════════════════════════════════════════════════
[MM-CNN] resuming from epoch 100/100
[MM-CNN] already complete, restoring best weights
[plot] /media/Ubunt_2/Project/pipeline_output/cnn_curves.png

───────────────────────────────────────────────────────
 Multimodal CNN
──────────────────────────────────

In [7]:
with open(os.path.join(OUT_DIR, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

In [8]:
# ── rebuild and load models ───────────────────────────────────────────────────
mm_cnn = MultimodalCNN(n_classes=3).to(DEVICE)
mm_cnn.load_state_dict(torch.load(os.path.join(OUT_DIR, "mm_cnn.pt"), map_location=DEVICE))
mm_cnn.eval()

ae = ConvAutoencoder(latent_dim=LATENT_DIM).to(DEVICE)
ae.load_state_dict(torch.load(os.path.join(OUT_DIR, "ae.pt"), map_location=DEVICE))
ae.eval()

fused = FusedClassifier(mm_cnn, ae, latent_dim=LATENT_DIM).to(DEVICE)
fused.load_state_dict(torch.load(os.path.join(OUT_DIR, "fused.pt"), map_location=DEVICE))
fused.eval()

FusedClassifier(
  (img_enc): ImageEncoder(
    (net): Sequential(
      (0): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Dropout2d(p=0.1, inplace=False)
      )
      (1): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2

In [9]:
# ── inference ─────────────────────────────────────────────────────────────────
def classify(img_path, u, g, r, i, z):
    tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    img = tf(Image.open(img_path).convert("RGB")).unsqueeze(0).to(DEVICE)

    raw = np.array([u, g, r, i, z, u-g, g-r, r-i, i-z], dtype=np.float32).reshape(1, -1)
    tab = torch.tensor(scaler.transform(raw), dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        cnn_probs = torch.softmax(mm_cnn(img, tab), dim=1).cpu().numpy()[0]
        fus_probs = torch.softmax(fused(img, tab),  dim=1).cpu().numpy()[0]

    print(f"\n  CNN   → {CLASSES[cnn_probs.argmax()]}  ({cnn_probs.max()*100:.1f}%)")
    print(f"  Fused → {CLASSES[fus_probs.argmax()]}  ({fus_probs.max()*100:.1f}%)")
    for cls, cp, fp in zip(CLASSES, cnn_probs, fus_probs):
        print(f"    {cls:<8}  CNN {cp*100:5.1f}%   Fused {fp*100:5.1f}%")

    return CLASSES[fus_probs.argmax()]

In [20]:

classify(
        img_path="/home/fadhla/Documents/School/Projects-1/Projects/Space body classification/sdss_dr17_dataset/STAR/1237645942906028070.jpg",
        u=0, g=0, r=0, i=0, z=0
    )


  CNN   → STAR  (100.0%)
  Fused → STAR  (100.0%)
    GALAXY    CNN   0.0%   Fused   0.0%
    QSO       CNN   0.0%   Fused   0.0%
    STAR      CNN 100.0%   Fused 100.0%


'STAR'